# 🌿 MindEase — Mental Wellness Chatbot
> **Enhanced Edition** | Emotion Detection · Mood Analytics · Crisis Safety

---
**Features:**
- 🎯 Multi-class emotion detection with confidence scores (7 emotions)
- 🆘 Automatic crisis keyword detection with helpline routing
- 🧠 DialoGPT conversational layer with memory management
- 📊 Structured mood logging (JSONL) for rich analytics
- 📈 Interactive Plotly mood dashboard
- 🧱 Fully modular, production-ready code structure

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
!pip install -q nltk transformers torch sentence-transformers
!pip install -q plotly pandas
print("✅ All packages installed.")

## 📦 Cell 2 — Imports & Configuration

In [ ]:
import sys
import random
import datetime
import json
import re
from pathlib import Path
from collections import Counter

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# ── Environment detection ─────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
USE_VOICE = not IN_COLAB  # Microphone & TTS disabled in Colab

LOG_FILE = Path("mood_log.jsonl")

print(f"Running in: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"Voice features: {'disabled' if not USE_VOICE else 'enabled'}")

## 💬 Cell 3 — Response Library & Crisis Detection

In [ ]:
# ── Emotion → Response mapping ─────────────────────────────────────────────────
CALMING_RESPONSES = {
    "sadness": [
        "It's completely okay to feel sad. I'm right here with you. 💚",
        "Sometimes emotions need space to breathe. I'm listening.",
        "You're not alone in this. Things get better, one moment at a time.",
        "Your feelings are real and they matter deeply.",
    ],
    "joy": [
        "That's wonderful! Hold onto that feeling — you deserve it. 😊",
        "I'm genuinely happy you're feeling this way!",
        "Celebrate those good moments — they're yours to keep.",
        "That spark in your words made me smile too!",
    ],
    "love": [
        "Love is such a powerful feeling. Thank you for sharing that. ❤️",
        "That's heartwarming. You deserve kindness and love in return.",
        "Cherish those connections — they're what makes life meaningful.",
    ],
    "anger": [
        "Your anger is valid. Tell me what happened.",
        "It's okay to be upset. Let's work through this together.",
        "Take a slow breath with me. I'm here and I'm listening.",
        "That sounds really frustrating. You have every right to feel that way.",
    ],
    "fear": [
        "Fear is natural and human. You're safe right now. 🌿",
        "Try grounding: name 5 things you can see around you right now.",
        "I believe in your strength, even when you can't feel it.",
        "You don't have to face this alone. I'm here.",
    ],
    "surprise": [
        "Wow, that sounds unexpected! How are you feeling about it?",
        "Life really does throw curveballs. Want to talk it through?",
        "That must have caught you off guard. Take your time.",
    ],
    "neutral": [
        "I'm here and listening. Tell me more about what's on your mind.",
        "How are you feeling today overall?",
        "I'm glad you're here. What's going on?",
    ],
}

STRESS_TIPS = [
    "Box breathing: inhale 4s → hold 4s → exhale 4s → hold 4s. Repeat 4 times.",
    "Drink a glass of water slowly. Hydration affects mood more than we think.",
    "Step outside for even 5 minutes. Fresh air resets the nervous system.",
    "Repeat quietly: 'I am calm. I am safe. This will pass.'",
    "Tense every muscle for 5 seconds, then release. Feel the relief.",
    "Put on one song you love and do nothing else — just listen fully.",
    "Write down 3 things you can control right now.",
]

# ── Crisis detection ──────────────────────────────────────────────────────────
CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "don't want to live",
    "want to die", "self harm", "hurt myself", "cutting", "no reason to live",
    "can't go on", "give up on life", "worthless", "hopeless", "nobody cares",
]

CRISIS_RESPONSE = """🆘 I'm really concerned about what you've shared, and I want you to know you matter deeply.

Please reach out to a crisis helpline right now:
  📞 iCall (India): 9152987821
  📞 Vandrevala Foundation: 1860-2662-345 (24/7, free)
  🌐 International: https://findahelpline.com

A trained counselor is ready to listen — you don't have to carry this alone. 💚"""

EMOTION_EMOJI = {
    "sadness": "😔", "joy": "😊", "love": "❤️",
    "anger": "😤", "fear": "😨", "surprise": "😮", "neutral": "😐",
}

def is_crisis(text: str) -> bool:
    text_lower = text.lower()
    return any(kw in text_lower for kw in CRISIS_KEYWORDS)

print("✅ Response library loaded.")

## 🤖 Cell 4 — Load AI Models

In [ ]:
print("Loading emotion classifier...")
emotion_classifier = pipeline(
    "text-classification",
    model="nateraw/bert-base-uncased-emotion",
    top_k=None,  # Return all emotion scores
)

print("Loading DialoGPT...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")
gpt_model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small")

print("✅ Models ready!")

## 🔧 Cell 5 — Core Functions

In [ ]:
# ── Emotion detection ─────────────────────────────────────────────────────────
def get_emotion(text: str):
    """Returns (top_emotion, confidence, all_scores_dict)."""
    results = emotion_classifier(text)
    scores = results[0] if isinstance(results[0], list) else results
    scores_dict = {item["label"]: round(item["score"], 3) for item in scores}
    top = max(scores, key=lambda x: x["score"])
    return top["label"], round(top["score"], 3), scores_dict


# ── Response generation ───────────────────────────────────────────────────────
def build_empathy_response(emotion: str, confidence: float) -> str:
    """Empathetic response + wellness tip for distress emotions."""
    base = random.choice(CALMING_RESPONSES.get(emotion, CALMING_RESPONSES["neutral"]))
    if emotion in ("sadness", "anger", "fear") and confidence > 0.5:
        tip = random.choice(STRESS_TIPS)
        base += f"\n\n💡 Tip: {tip}"
    return base


def gpt_response(user_input: str, chat_history_ids=None):
    """Generate conversational GPT reply with memory & repetition guard."""
    input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt")
    if chat_history_ids is not None:
        combined = torch.cat([chat_history_ids, input_ids], dim=-1)
        # Sliding window — keep only last 512 tokens
        if combined.shape[-1] > 512:
            combined = combined[:, -512:]
        input_ids = combined
    output = gpt_model.generate(
        input_ids,
        max_length=min(input_ids.shape[-1] + 80, 600),
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.92,
        temperature=0.85,
        repetition_penalty=1.3,
    )
    reply = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
    return reply.strip() or "I'm here with you.", output


# ── Mood logging ──────────────────────────────────────────────────────────────
def log_emotion(text: str, emotion: str, confidence: float, scores: dict):
    """Append structured JSONL entry for rich analytics."""
    entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "message": text,
        "emotion": emotion,
        "confidence": confidence,
        "all_scores": scores,
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")


def load_mood_log():
    if not LOG_FILE.exists():
        return []
    entries = []
    with open(LOG_FILE) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    entries.append(json.loads(line))
                except:
                    pass
    return entries


# ── Voice (local Jupyter only) ────────────────────────────────────────────────
def speak(text: str):
    if not USE_VOICE:
        return
    try:
        import pyttsx3
        engine = pyttsx3.init()
        engine.say(text)
        engine.runAndWait()
    except Exception:
        pass


def get_user_input(prompt="🗣 You: "):
    if not USE_VOICE:
        return input(prompt)
    try:
        import speech_recognition as sr
        r = sr.Recognizer()
        with sr.Microphone() as src:
            print("🎤 Listening...")
            audio = r.listen(src, timeout=5)
        return r.recognize_google(audio)
    except Exception as e:
        print(f"⚠️ Voice error: {e}")
        return input(prompt)

print("✅ Core functions ready.")

## 🌿 Cell 6 — Run the Chatbot

In [ ]:
def chatbot():
    print("═" * 60)
    print("  🌿 MindEase — Mental Wellness Chatbot")
    print("  Type 'bye' to exit · 'log' to see mood history")
    print("═" * 60)

    chat_history_ids = None
    session_emotions = []

    while True:
        print()
        user_input = get_user_input("🗣 You: ")

        if not user_input.strip():
            continue

        # ── Special commands ──
        if user_input.lower() in ("bye", "exit", "quit"):
            print("\n🤖 Bot: Take care — you're not alone. Come back anytime. 💚")
            speak("Take care, you are not alone. Come back anytime.")
            if session_emotions:
                freq = Counter(session_emotions).most_common(1)[0][0]
                print(f"\n📊 Session summary: Dominant mood was '{freq}' {EMOTION_EMOJI.get(freq,'')}")
            break

        if user_input.lower() == "log":
            entries = load_mood_log()
            print(f"\n📋 {len(entries)} mood entries logged to {LOG_FILE}")
            continue

        # ── Crisis check (FIRST, before anything else) ──
        if is_crisis(user_input):
            print(f"\n🤖 Bot:\n{CRISIS_RESPONSE}")
            speak("I'm really concerned. Please reach out to a crisis helpline right now.")
            log_emotion(user_input, "crisis", 1.0, {})
            continue

        # ── Emotion analysis ──
        emotion, confidence, scores = get_emotion(user_input)
        session_emotions.append(emotion)

        # ── Generate responses ──
        empathy_reply = build_empathy_response(emotion, confidence)
        gpt_reply, chat_history_ids = gpt_response(user_input, chat_history_ids)

        # ── Display ──
        emoji = EMOTION_EMOJI.get(emotion, "🤖")
        print(f"\n🤖 Bot [{emoji} {emotion} · {confidence:.0%}]:")
        print(f"{empathy_reply}")
        if gpt_reply and len(gpt_reply) > 5:
            print(f"\n   💭 {gpt_reply}")

        # ── Show top-3 emotions ──
        top3 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:3]
        bar = " | ".join([f"{EMOTION_EMOJI.get(e,'')} {e}: {s:.0%}" for e, s in top3])
        print(f"\n   📊 {bar}")

        speak(empathy_reply)
        log_emotion(user_input, emotion, confidence, scores)


# ── Run ──────────────────────────────────────────────────────────────────────
chatbot()

## 📊 Cell 7 — Mood Analytics Dashboard

In [ ]:
def show_mood_dashboard():
    entries = load_mood_log()
    if not entries:
        print("No mood data yet. Run the chatbot first!")
        return

    df = pd.DataFrame(entries)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["date"] = df["timestamp"].dt.date
    df["hour"] = df["timestamp"].dt.hour

    # ── Summary stats ──────────────────────────────────────────────────────
    counts = df["emotion"].value_counts()
    print("═" * 50)
    print("  📊 Mood Analytics Summary")
    print("═" * 50)
    print(f"  Total sessions : {len(df)}")
    print(f"  Most frequent  : {counts.index[0]} {EMOTION_EMOJI.get(counts.index[0], '')}")
    print(f"  Date range     : {df['date'].min()} → {df['date'].max()}")
    print()
    for emotion, count in counts.items():
        pct = count / len(df) * 100
        bar = "█" * int(pct / 5)
        print(f"  {EMOTION_EMOJI.get(emotion, '  ')} {emotion:<10} {bar} {pct:.1f}%")

    # ── Chart 1: Donut ─────────────────────────────────────────────────────
    EMOTION_COLOR = {
        "sadness": "#6ba3d6", "joy": "#f9c74f", "love": "#f4a0a0",
        "anger": "#e05252", "fear": "#9b72cf", "surprise": "#f3a04a", "neutral": "#8ab4a1",
    }
    colors = [EMOTION_COLOR.get(e, "#cccccc") for e in counts.index]

    fig1 = go.Figure(go.Pie(
        labels=[f"{EMOTION_EMOJI.get(e,'')} {e}" for e in counts.index],
        values=counts.values,
        hole=0.5,
        marker_colors=colors,
    ))
    fig1.update_layout(title="Emotion Breakdown", height=400)
    fig1.show()

    # ── Chart 2: Stacked bar over time ──────────────────────────────────────
    daily = df.groupby(["date", "emotion"]).size().reset_index(name="count")
    fig2 = px.bar(
        daily, x="date", y="count", color="emotion",
        color_discrete_map=EMOTION_COLOR,
        title="Mood Over Time",
        labels={"count": "Messages", "date": "Date"},
    )
    fig2.show()

    # ── Chart 3: Hour-of-day activity ───────────────────────────────────────
    hour_counts = df.groupby("hour").size().reset_index(name="sessions")
    fig3 = px.bar(
        hour_counts, x="hour", y="sessions",
        title="When Do You Check In?",
        labels={"hour": "Hour of Day", "sessions": "Sessions"},
        color="sessions", color_continuous_scale="Greens",
    )
    fig3.show()

    # ── Confidence over time ─────────────────────────────────────────────────
    fig4 = px.line(
        df.sort_values("timestamp"),
        x="timestamp", y="confidence", color="emotion",
        color_discrete_map=EMOTION_COLOR,
        title="Emotion Confidence Over Time",
        labels={"confidence": "Model Confidence", "timestamp": "Time"},
        markers=True,
    )
    fig4.show()


show_mood_dashboard()

---
## 🌐 Cell 8 — Launch Streamlit Web App (Optional)
Run this cell to launch the full web UI with chat + dashboard.

In [ ]:
# Uncomment and run in Colab to launch the web app:
# !pip install -q streamlit pyngrok
# from pyngrok import ngrok
# !streamlit run app.py &
# public_url = ngrok.connect(8501)
# print("Web app live at:", public_url)